# Lilac API Reference

**Lilac** is an online epub/pdf reader powered by agents. This notebook documents the backend API: its response schema, data models, endpoints, and core features.

| | |
|---|---|
| **Base URL** | `http://localhost:8000` |
| **Framework** | FastAPI (Python) |
| **Database** | SQLite via SQLAlchemy |
| **Vector Store** | ChromaDB + Ollama (`nomic-embed-text`) |
| **Supported formats** | `.epub`, `.pdf` |

## Table of Contents
1. [Standardized Response Schema](#1-standardized-response-schema)
2. [Data Models](#2-data-models)
3. [Endpoints](#3-endpoints)
   - [3.1 Health](#31-health)
   - [3.2 Files](#32-files)
   - [3.3 Planned Endpoints](#33-planned-endpoints)
4. [Feature: File Parsing](#4-feature-file-parsing)
5. [Feature: Vector Store](#5-feature-vector-store)
6. [Feature: Database](#6-feature-database)

---
## 1. Standardized Response Schema

Every endpoint wraps its payload in a consistent envelope. This makes it trivial for clients to determine success or failure without inspecting HTTP status codes alone.

### Success
```json
{
  "response": <data>,
  "error": {}
}
```

### Error
```json
{
  "response": {},
  "error": {
    "message": "<human-readable description>",
    "code": <HTTP status code>
  }
}
```

The `error.code` mirrors the HTTP response status code (e.g. `404`, `415`, `422`, `500`).

In [ ]:
# models.py — response envelope helpers
from typing import Any
from fastapi.responses import JSONResponse

def ok(data: Any) -> dict:
    """Wrap a successful payload."""
    return {"response": data, "error": {}}

def err(msg: str, code: int = 400) -> JSONResponse:
    """Return a JSON error response."""
    return JSONResponse(
        content={"response": {}, "error": {"message": msg, "code": code}},
        status_code=code,
    )

# Examples
import json
print("Success:", json.dumps(ok({"id": "abc", "deleted": True}), indent=2))
print("\nError payload:", json.dumps(
    {"response": {}, "error": {"message": "File not found", "code": 404}}, indent=2
))

---
## 2. Data Models

### FileRecord
Returned by all `/files` endpoints.

| Field | Type | Description |
|---|---|---|
| `id` | `string` (UUID) | Unique identifier |
| `filename` | `string` | Original filename as uploaded |
| `file_type` | `"epub"` \| `"pdf"` | Detected format |
| `size_bytes` | `integer` | Raw file size in bytes |
| `title` | `string` | Extracted from document metadata |
| `author` | `string` | Extracted from document metadata |
| `chunk_count` | `integer` | Number of text chunks indexed in the vector store |
| `uploaded_at` | `string` (ISO-8601 UTC) | Upload timestamp |

### LoginRequest / RegisterRequest *(planned)*

| Field | Type |
|---|---|
| `email` | `string` |
| `password` | `string` |

In [ ]:
# models.py — Pydantic schemas
from pydantic import BaseModel

class FileRecord(BaseModel):
    id: str
    filename: str
    file_type: str        # "epub" | "pdf"
    size_bytes: int
    title: str
    author: str
    chunk_count: int
    uploaded_at: str      # ISO-8601 UTC

class LoginRequest(BaseModel):    # planned
    email: str
    password: str

class RegisterRequest(BaseModel): # planned
    email: str
    password: str

# Example FileRecord payload
example = FileRecord(
    id="550e8400-e29b-41d4-a716-446655440000",
    filename="moby_dick.epub",
    file_type="epub",
    size_bytes=1048576,
    title="Moby Dick",
    author="Herman Melville",
    chunk_count=312,
    uploaded_at="2026-04-16T10:30:00+00:00",
)
print(example.model_dump_json(indent=2))

---
## 3. Endpoints

### 3.1 Health

#### `GET /health`
Liveness check — returns immediately, no dependencies queried.

**Response** `200 OK`
```json
{ "status": "ok" }
```

> Note: this endpoint does **not** use the standard envelope — it returns a bare object.

### 3.2 Files

All file endpoints are prefixed with `/files`.

---

#### `POST /files/upload`
Upload an epub or pdf file. On success the file is:
1. Saved to `uploads/<uuid>.<ext>`
2. Parsed to extract metadata + text chunks
3. Indexed into the vector store (ChromaDB)
4. Persisted to SQLite

**Request** — `multipart/form-data`

| Field | Type | Required |
|---|---|---|
| `file` | binary | yes |

**Response** `201 Created`
```json
{
  "response": {
    "id": "550e8400-e29b-41d4-a716-446655440000",
    "filename": "moby_dick.epub",
    "file_type": "epub",
    "size_bytes": 1048576,
    "title": "Moby Dick",
    "author": "Herman Melville",
    "chunk_count": 312,
    "uploaded_at": "2026-04-16T10:30:00+00:00"
  },
  "error": {}
}
```

**Errors**

| Code | Condition |
|---|---|
| `415` | File extension not `epub` or `pdf` |
| `422` | Parsing failed (corrupt or unreadable file) |
| `500` | Vector store ingestion failed |

---

#### `GET /files`
Return all uploaded files, newest first.

**Response** `200 OK`
```json
{
  "response": [
    { /* FileRecord */ },
    { /* FileRecord */ }
  ],
  "error": {}
}
```

---

#### `GET /files/{file_id}`
Fetch metadata for a single file.

**Path params**

| Param | Type |
|---|---|
| `file_id` | `string` (UUID) |

**Response** `200 OK`
```json
{
  "response": { /* FileRecord */ },
  "error": {}
}
```

**Errors**

| Code | Condition |
|---|---|
| `404` | No file with that id |

---

#### `DELETE /files/{file_id}`
Permanently remove a file from disk, the vector store, and the database.

**Path params**

| Param | Type |
|---|---|
| `file_id` | `string` (UUID) |

**Response** `200 OK`
```json
{
  "response": { "id": "550e8400-...", "deleted": true },
  "error": {}
}
```

**Errors**

| Code | Condition |
|---|---|
| `404` | No file with that id |

### 3.3 Planned Endpoints

These routes are scaffolded in `main.py` but not yet implemented.

#### Reader
| Method | Path | Description |
|---|---|---|
| `GET` | `/files/{id}/pages/{page}` | Return rendered page content for in-app reading |

#### Auth
| Method | Path | Description |
|---|---|---|
| `POST` | `/auth/register` | Create a new user account |
| `POST` | `/auth/login` | Authenticate and receive a session token |
| `POST` | `/auth/logout` | Invalidate the current session |
| `GET` | `/auth/me` | Return the authenticated user's profile |

---
## 4. Feature: File Parsing

Both parsers share the same chunking strategy and return the same tuple: `(title, author, chunks)`.

| Parameter | Value |
|---|---|
| Chunk size | 2 000 characters (~500 tokens) |
| Chunk overlap | 200 characters |

### EPUB — `parsers/epub.py`
Uses **ebooklib** to read the EPUB container and **BeautifulSoup** to strip HTML tags from each document item (chapter). Metadata (`DC:title`, `DC:creator`) is read from the EPUB manifest.

**Chunk shape**
```python
{
    "text": str,          # raw plain text segment
    "chapter": str,       # EPUB item name (e.g. "chapter01.xhtml")
    "chunk_index": int,   # global sequential index across all chapters
}
```

### PDF — `parsers/pdf.py`
Uses **PyMuPDF** (`fitz`) to extract text page-by-page. Metadata (`title`, `author`) is read from the PDF document properties.

**Chunk shape**
```python
{
    "text": str,          # raw plain text segment
    "page": int,          # 1-indexed page number
    "chunk_index": int,   # global sequential index across all pages
}
```

In [ ]:
# Chunking algorithm shared by both parsers
def _chunk_text(text: str, chunk_size: int = 2000, overlap: int = 200) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# Demo
sample = "A" * 5000
result = _chunk_text(sample)
print(f"Input length : {len(sample)}")
print(f"Chunks       : {len(result)}")
print(f"Chunk sizes  : {[len(c) for c in result]}")

---
## 5. Feature: Vector Store

The vector store enables semantic search over any uploaded document.

| | |
|---|---|
| **Engine** | ChromaDB (`PersistentClient`) |
| **Storage path** | `./chroma_db` |
| **Embedding model** | `nomic-embed-text` via Ollama |
| **Ollama URL** | `http://localhost:11434` |
| **Collection naming** | `file_{file_id}` |

### API surface (`vector_store.py`)

| Function | Signature | Description |
|---|---|---|
| `ingest` | `(file_id: str, chunks: list[dict]) -> None` | Embed and store all chunks for a file |
| `query` | `(file_id: str, text: str, n: int = 5) -> list[dict]` | Semantic nearest-neighbour search |
| `delete_collection` | `(file_id: str) -> None` | Remove all vectors for a file |

### Query result shape
```python
[
    {
        "text": str,       # matched chunk text
        "metadata": dict,  # chunk metadata (chapter/page + chunk_index)
        "distance": float, # L2 distance — lower is more similar
    },
    ...
]
```

In [ ]:
# vector_store.py — public interface (reference)
import chromadb
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction

_client = chromadb.PersistentClient(path="./chroma_db")
_embed_fn = OllamaEmbeddingFunction(
    url="http://localhost:11434",
    model_name="nomic-embed-text",
)

def ingest(file_id: str, chunks: list[dict]) -> None:
    collection = _client.get_or_create_collection(
        name=f"file_{file_id}", embedding_function=_embed_fn
    )
    collection.add(
        ids=[f"{file_id}_{c['chunk_index']}" for c in chunks],
        documents=[c["text"] for c in chunks],
        metadatas=[{k: v for k, v in c.items() if k != "text"} for c in chunks],
    )

def query(file_id: str, text: str, n: int = 5) -> list[dict]:
    collection = _client.get_collection(
        name=f"file_{file_id}", embedding_function=_embed_fn
    )
    results = collection.query(query_texts=[text], n_results=n)
    return [
        {"text": doc, "metadata": results["metadatas"][0][i], "distance": results["distances"][0][i]}
        for i, doc in enumerate(results["documents"][0])
    ]

def delete_collection(file_id: str) -> None:
    try:
        _client.delete_collection(name=f"file_{file_id}")
    except Exception:
        pass

---
## 6. Feature: Database

File metadata is persisted in a local SQLite database (`lilac.db`) managed through SQLAlchemy.

### `files` table — `FileRow`

| Column | SQLAlchemy type | Notes |
|---|---|---|
| `id` | `String` (PK) | UUID assigned at upload |
| `filename` | `String` | Original filename |
| `file_type` | `String` | `"epub"` or `"pdf"` |
| `size_bytes` | `Integer` | Raw byte count |
| `title` | `String` | From document metadata |
| `author` | `String` | From document metadata |
| `chunk_count` | `Integer` | Chunks ingested into vector store |
| `uploaded_at` | `DateTime` | UTC timestamp |
| `path` | `String` | Absolute path to the saved file on disk |

In [ ]:
# database.py — schema and session helper
from sqlalchemy import create_engine, Column, String, Integer, DateTime
from sqlalchemy.orm import DeclarativeBase, Session
from datetime import datetime, timezone
from contextlib import contextmanager

engine = create_engine("sqlite:///./lilac.db", connect_args={"check_same_thread": False})

class Base(DeclarativeBase):
    pass

class FileRow(Base):
    __tablename__ = "files"

    id          = Column(String,  primary_key=True)
    filename    = Column(String,  nullable=False)
    file_type   = Column(String,  nullable=False)   # "epub" | "pdf"
    size_bytes  = Column(Integer, nullable=False)
    title       = Column(String,  nullable=False, default="")
    author      = Column(String,  nullable=False, default="")
    chunk_count = Column(Integer, nullable=False, default=0)
    uploaded_at = Column(DateTime, nullable=False,
                         default=lambda: datetime.now(timezone.utc))
    path        = Column(String,  nullable=False)

def get_session() -> Session:
    return Session(engine)

def init_db() -> None:
    Base.metadata.create_all(bind=engine)